In [3]:
#get all the tables:
from db_setup import conn, currencyexchange, customer, date, product, sales, store
#instead of saving csv files in sql and then importing one by one
import pandas as pd

customer_summary=pd.read_parquet(r"C:\Users\amyre\Desktop\Luke Barousse SQL Course\SQL + Contoso\saved_files\customer_summary.parquet")


In [4]:
#Get the cut-off values for the percentile segments:

p25=customer_summary["lifetime_profit"].quantile(0.25)
p50=customer_summary["lifetime_profit"].quantile(0.50)
p75=customer_summary["lifetime_profit"].quantile(0.75)

print(f'p25={p25:.2f}, p50={p50:.2f}, p75={p75:.2f}')


p25=433.82, p50=1295.47, p75=3039.32


In [5]:
#Build the case segments:

def value_segment(p):
    if p>=p75:
        return "High Value"
    if p>=p50:
        return "Mid-High Value"
    if p>=p25:
        return "Low-Mid Value"
    else:
        return "Low Value"
    
customer_summary["value_segment"]=customer_summary["lifetime_profit"].apply(value_segment)

In [ ]:
#Saving the customer_value_segments for later use:

customer_value_segments=customer_summary[["customerkey","lifetime_revenue","lifetime_profit","order_count","value_segment"]].copy()
customer_value_segments.to_parquet(r"C:\Users\amyre\Desktop\Luke Barousse SQL Course\SQL + Contoso\saved_files\customer_value_segments.parquet", index=False)

customer_value_segments



,customerkey,lifetime_revenue,lifetime_profit,order_count,value_segment
0,15,1299.708307,663.841613,1,Low-Mid Value
1,180,1103.346104,548.366791,2,Low-Mid Value
2,185,666.453820,280.348295,1,Low Value
3,243,148.903542,66.399387,1,Low Value
4,387,2341.268364,1217.535628,3,Low-Mid Value
...,...,...,...,...,...
49482,2099619,6709.935970,3943.559170,4,High Value
49483,2099656,10404.677800,5485.587800,4,High Value
49484,2099697,38.201100,20.371100,1,Low Value
49485,2099711,6008.670000,2989.910000,2,Mid-High Value


In [8]:
segment_summary=(
    customer_summary
    .groupby("value_segment")
    .agg(
        customers=("customerkey","count"),
        avg_profit=("lifetime_profit","mean"),
        total_profit=("lifetime_profit","sum"),
        avg_orders=("order_count","mean")
    )
    .reset_index()
)

segment_summary["profit_share"]=segment_summary["total_profit"]/(segment_summary["total_profit"].sum()) 


segment_summary.sort_values("total_profit", ascending=False)



,value_segment,customers,avg_profit,total_profit,avg_orders,profit_share
0,High Value,12372,6279.465702,7.768955e+07,2.337375,0.673339
3,Mid-High Value,12372,2043.592967,2.528333e+07,1.796233,0.219132
2,Low-Mid Value,12371,825.148907,1.020792e+07,1.445396,0.088472
1,Low Value,12372,177.724300,2.198805e+06,1.140317,0.019057
